# 优化提示

在使用 AI 模型时，编写有效的提示是一项关键技能。即使是经验丰富的用户，也可能在无意中引入矛盾、歧义或不一致之处，导致结果不够理想。这里演示的系统可以帮助识别并修复常见问题，从而得到更可靠、更有效的提示。

优化过程采用多智能体方法，由多个专门的 AI 智能体协作分析和重写提示。系统会自动识别并处理以下几类常见问题：

- 提示指令中的**矛盾**
- 缺失或不清晰的**格式规范**
- 提示与少样本示例之间的**不一致**

---

**目标**：本 cookbook 演示了将 Agents SDK 与 Evals 结合使用的最佳实践，用以构建 OpenAI 提示优化系统的早期版本。你可以使用这段代码优化提示，也可以使用[我们 playground 中的优化器！](https://platform.openai.com/playground/prompts)








Ask ChatGPT


**Cookbook 结构**  
本 notebook 的结构如下：

- [第 1 步：系统概览](#1-系统概览) - 了解提示优化系统的工作原理  
- [第 2 步：数据模型](#2-数据模型) - 了解系统使用的数据结构
- [第 3 步：定义智能体](#3-定义智能体) - 查看分析和改进提示的智能体
- [第 4 步：评估](#4-使用评估来确定这些智能体) - 使用 Evals 验证我们对智能体模型和指令的选择
- [第 5 步：运行优化工作流](#5-运行优化工作流) - 了解工作流如何传递提示
- [第 6 步：示例](#6-示例) - 探索提示优化的真实示例

**前提条件**
- `openai` Python 包 
- `openai-agents` 包
- 在环境变量中设置为 `OPENAI_API_KEY` 的 OpenAI API 密钥

## 1. 系统概览

提示优化系统采用协作式多智能体方法来分析和改进提示。每个智能体专门负责检测或重写某一类特定问题：

1. **Dev-Contradiction-Checker**：扫描提示中的逻辑矛盾或无法执行的指令，例如同一个提示中同时出现"只使用正数"和"包含负数示例"。

2. **Format-Checker**：识别提示期望结构化输出（如 JSON、CSV 或 Markdown）却没有清楚说明确切格式要求的情况。该智能体确保所有必要的字段、数据类型和格式规则都得到明确定义。

3. **Few-Shot-Consistency-Checker**：检查示例对话，确保助手的回复确实遵循提示中规定的规则。它能发现提示要求与示例演示之间的不匹配。

4. **Dev-Rewriter**：在识别出问题后，该智能体会重写提示，在保留原始意图的同时解决矛盾并澄清格式规范。

5. **Few-Shot-Rewriter**：更新不一致的示例回复，使其与提示中的规则保持一致，确保所有示例都正确遵循新的开发者提示。

通过协同工作，这些智能体可以系统地识别并修复提示中的问题。

In [ ]:
# Import required modules
from openai import AsyncOpenAI
import asyncio
import json
import os
from enum import Enum
from typing import Any, List, Dict
from pydantic import BaseModel, Field
from agents import Agent, Runner, set_default_openai_client, trace

openai_client: AsyncOpenAI | None = None

def _get_openai_client() -> AsyncOpenAI:
    global openai_client
    if openai_client is None:
        openai_client = AsyncOpenAI(
            api_key=os.environ.get("OPENAI_API_KEY", "your-api-key"),
        )
    return openai_client

set_default_openai_client(_get_openai_client())

## 2. 数据模型

为了便于智能体之间进行结构化通信，系统使用 Pydantic 模型来定义输入和输出的预期格式。这些 Pydantic 模型有助于验证数据，并确保整个工作流的一致性。

数据模型包括：

1. **Role** - 消息角色（user/assistant）的枚举
2. **ChatMessage** - 表示对话中的单条消息
3. **Issues** - 用于报告检测到的问题的基础模型
4. **FewShotIssues** - 扩展模型，增加了针对示例消息的重写建议
5. **MessagesOutput** - 包含优化后的对话消息
6. **DevRewriteOutput** - 包含改进后的开发者提示

使用 Pydantic 可以让系统在流程的每一步验证所有数据是否符合预期格式。

In [32]:
class Role(str, Enum):
    """Role enum for chat messages."""
    user = "user"
    assistant = "assistant"

class ChatMessage(BaseModel):
    """Single chat message used in few-shot examples."""
    role: Role
    content: str

class Issues(BaseModel):
    """Structured output returned by checkers."""
    has_issues: bool
    issues: List[str]
    
    @classmethod
    def no_issues(cls) -> "Issues":
        return cls(has_issues=False, issues=[])

class FewShotIssues(Issues):
    """Output for few-shot contradiction detector including optional rewrite suggestions."""
    rewrite_suggestions: List[str] = Field(default_factory=list)
    
    @classmethod
    def no_issues(cls) -> "FewShotIssues":
        return cls(has_issues=False, issues=[], rewrite_suggestions=[])

class MessagesOutput(BaseModel):
    """Structured output returned by `rewrite_messages_agent`."""

    messages: list[ChatMessage]


class DevRewriteOutput(BaseModel):
    """Rewriter returns the cleaned-up developer prompt."""

    new_developer_message: str

## 3. 定义智能体

在本节中，我们使用 `openai-agents` 包中的 `Agent` 类创建专门的 AI 智能体。从这些智能体定义中可以看出编写有效 AI 指令的几项最佳实践：

### 智能体指令的最佳实践

1. **明确的范围定义**：每个智能体都有一个边界清晰、范围狭窄的目标。例如，矛盾检查器只关注"真正的自相矛盾"，并明确指出"重叠或冗余不属于矛盾"。

2. **分步流程**：指令提供了清晰的方法论，例如格式检查器会先对任务进行分类，然后再分析格式要求。

3. **明确的定义**：关键术语都有精确定义以避免歧义。少样本一致性检查器包含一份详细的"合规评分标准"（Compliance Rubric），准确解释了什么才算合规。

4. **设定边界**：指令会说明智能体不应该做什么。少样本检查器明确列出了哪些内容"超出范围"，以防过度标记问题。

5. **结构化输出要求**：每个智能体都有严格定义的输出格式并附有示例，确保优化流水线的一致性。

这些原则造就了可靠、专注的智能体，使它们能在优化系统中高效协作。下面是完整的智能体定义及其详细指令。

In [ ]:
dev_contradiction_checker = Agent(
    name="contradiction_detector",
    model="gpt-4.1",
    output_type=Issues,
    instructions="""
    You are **Dev-Contradiction-Checker**.

    Goal
    Detect *genuine* self-contradictions or impossibilities **inside** the developer prompt supplied in the variable `DEVELOPER_MESSAGE`.

    Definitions
    • A contradiction = two clauses that cannot both be followed.
    • Overlaps or redundancies in the DEVELOPER_MESSAGE are *not* contradictions.

    What you MUST do
    1. Compare every imperative / prohibition against all others.
    2. List at most FIVE contradictions (each as ONE bullet).
    3. If no contradiction exists, say so.

    Output format (**strict JSON**)
    Return **only** an object that matches the `Issues` schema:

    ```json
    {"has_issues": <bool>,
    "issues": [
        "<bullet 1>",
        "<bullet 2>"
    ]
    }
    - has_issues = true IFF the issues array is non-empty.
    - Do not add extra keys, comments or markdown.
""",
)
format_checker = Agent(
    name="format_checker",
    model="gpt-4.1",
    output_type=Issues,
    instructions="""
    You are Format-Checker.

    Task
    Decide whether the developer prompt requires a structured output (JSON/CSV/XML/Markdown table, etc.).
    If so, flag any missing or unclear aspects of that format.

    Steps
    Categorise the task as:
    a. "conversation_only", or
    b. "structured_output_required".

    For case (b):
    - Point out absent fields, ambiguous data types, unspecified ordering, or missing error-handling.

    Do NOT invent issues if unsure. be a little bit more conservative in flagging format issues

    Output format
    Return strictly-valid JSON following the Issues schema:

    {
    "has_issues": <bool>,
    "issues": ["<desc 1>", "..."]
    }
    Maximum five issues. No extra keys or text.
""",
)
fewshot_consistency_checker = Agent(
    name="fewshot_consistency_checker",
    model="gpt-4.1",
    output_type=FewShotIssues,
    instructions="""
    You are FewShot-Consistency-Checker.

    Goal
    Find conflicts between the DEVELOPER_MESSAGE rules and the accompanying **assistant** examples.

    USER_EXAMPLES:      <all user lines>          # context only
    ASSISTANT_EXAMPLES: <all assistant lines>     # to be evaluated

    Method
    Extract key constraints from DEVELOPER_MESSAGE:
    - Tone / style
    - Forbidden or mandated content
    - Output format requirements

    Compliance Rubric - read carefully
    Evaluate only what the developer message makes explicit.

    Objective constraints you must check when present:
    - Required output type syntax (e.g., "JSON object", "single sentence", "subject line").
    - Hard limits (length ≤ N chars, language required to be English, forbidden words, etc.).
    - Mandatory tokens or fields the developer explicitly names.

    Out-of-scope (DO NOT FLAG):
    - Whether the reply "sounds generic", "repeats the prompt", or "fully reflects the user's request" - unless the developer text explicitly demands those qualities.
    - Creative style, marketing quality, or depth of content unless stated.
    - Minor stylistic choices (capitalisation, punctuation) that do not violate an explicit rule.

    Pass/Fail rule
    - If an assistant reply satisfies all objective constraints, it is compliant, even if you personally find it bland or loosely related.
    - Only record an issue when a concrete, quoted rule is broken.

    Empty assistant list ⇒ immediately return has_issues=false.

    For each assistant example:
    - USER_EXAMPLES are for context only; never use them to judge compliance.
    - Judge each assistant reply solely against the explicit constraints you extracted from the developer message.
    - If a reply breaks a specific, quoted rule, add a line explaining which rule it breaks.
    - Optionally, suggest a rewrite in one short sentence (add to rewrite_suggestions).
    - If you are uncertain, do not flag an issue.
    - Be conservative—uncertain or ambiguous cases are not issues.

    be a little bit more conservative in flagging few shot contradiction issues
    Output format
    Return JSON matching FewShotIssues:

    {
    "has_issues": <bool>,
    "issues": ["<explanation 1>", "..."],
    "rewrite_suggestions": ["<suggestion 1>", "..."] // may be []
    }
    List max five items for both arrays.
    Provide empty arrays when none.
    No markdown, no extra keys.
    """,
)
dev_rewriter = Agent(
    name="dev_rewriter",
    model="gpt-4.1",
    output_type=DevRewriteOutput,
    instructions="""
    You are Dev-Rewriter.

    You receive:
    - ORIGINAL_DEVELOPER_MESSAGE
    - CONTRADICTION_ISSUES (may be empty)
    - FORMAT_ISSUES (may be empty)

    Rewrite rules
    Preserve the original intent and capabilities.

    Resolve each contradiction:
    - Keep the clause that preserves the message intent; remove/merge the conflicting one.

    If FORMAT_ISSUES is non-empty:
    - Append a new section titled ## Output Format that clearly defines the schema or gives an explicit example.

    Do NOT change few-shot examples.

    Do NOT add new policies or scope.

    Output format (strict JSON)
    {
    "new_developer_message": "<full rewritten text>"
    }
    No other keys, no markdown.
""",
)
fewshot_rewriter = Agent(
    name="fewshot_rewriter",
    model="gpt-4.1",
    output_type=MessagesOutput,
    instructions="""
    You are FewShot-Rewriter.

    Input payload
    - NEW_DEVELOPER_MESSAGE (already optimized)
    - ORIGINAL_MESSAGES (list of user/assistant dicts)
    - FEW_SHOT_ISSUES (non-empty)

    Task
    Regenerate only the assistant parts that were flagged.
    User messages must remain identical.
    Every regenerated assistant reply MUST comply with NEW_DEVELOPER_MESSAGE.

    After regenerating each assistant reply, verify:
    - It matches NEW_DEVELOPER_MESSAGE. ENSURE THAT THIS IS TRUE.

    Output format
    Return strict JSON that matches the MessagesOutput schema:

    {
    "messages": [
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "..."}
        ]
    }
    Guidelines
    - Preserve original ordering and total count.
    - If a message was unproblematic, copy it unchanged.
    """,
)

## 4. 使用评估来确定这些智能体

我们来看看如何使用 OpenAI Evals 调整智能体指令并选择合适的模型。为此，我们构建了一组黄金示例：每个示例都包含原始消息（开发者消息 + user/assistant 消息）以及优化工作流应做出的修改。以下是我们使用的两个黄金示例对：

```
[
  {
    "focus": "contradiction_issues",
    "input_payload": {
      "developer_message": "Always answer in **English**.\nNunca respondas en inglés.",
      "messages": [
        {
          "role": "user",
          "content": "¿Qué hora es?"
        }
      ]
    },
    "golden_output": {
      "changes": true,
      "new_developer_message": "Always answer **in English**.",
      "new_messages": [
        {
          "role": "user",
          "content": "¿Qué hora es?"
        }
      ],
      "contradiction_issues": "Developer message simultaneously insists on English and forbids it.",
      "few_shot_contradiction_issues": "",
      "format_issues": "",
      "general_improvements": ""
    }
  },
  {
    "focus": "few_shot_contradiction_issues",
    "input_payload": {
      "developer_message": "Respond with **only 'yes' or 'no'** – no explanations.",
      "messages": [
        {
          "role": "user",
          "content": "Is the sky blue?"
        },
        {
          "role": "assistant",
          "content": "Yes, because wavelengths …"
        },
        {
          "role": "user",
          "content": "Is water wet?"
        },
        {
          "role": "assistant",
          "content": "Yes."
        }
      ]
    },
    "golden_output": {
      "changes": true,
      "new_developer_message": "Respond with **only** the single word \"yes\" or \"no\".",
      "new_messages": [
        {
          "role": "user",
          "content": "Is the sky blue?"
        },
        {
          "role": "assistant",
          "content": "yes"
        },
        {
          "role": "user",
          "content": "Is water wet?"
        },
        {
          "role": "assistant",
          "content": "yes"
        }
      ],
      "contradiction_issues": "",
      "few_shot_contradiction_issues": "Assistant examples include explanations despite instruction not to.",
      "format_issues": "",
      "general_improvements": ""
    }
  }
]
```

我们手工标注了 20 个黄金输出，涵盖矛盾问题、少样本问题、格式问题、无问题以及多种问题的组合。基于这些数据，我们构建了一个 python 字符串检查评分器来验证两件事：每个黄金示例对是否检测到了问题，以及检测到的问题是否与预期一致。根据这一信号，我们调整了智能体指令和所用模型，以最大化在该评估上的准确率。最终，我们选择了 4.1 模型，在准确率、成本和速度之间取得平衡。我们使用的具体提示也遵循了 4.1 提示指南。可以看到，我们在全部 20 个黄金输出上都得到了正确的标签：既识别出了正确的问题，又避免了误报。 

![黄金集上的准确率](../images/optimizepromptfig1.png)

![黄金集上的评估](../images/optimizepromptfig2.png)

## 5. 运行优化工作流

我们来深入了解优化系统端到端的实际运作方式。核心工作流由多个并行运行的智能体组成，以高效地处理和优化提示。

In [ ]:
def _normalize_messages(messages: List[Any]) -> List[Dict[str, str]]:
    """Convert list of pydantic message models to JSON-serializable dicts."""
    result = []
    for m in messages:
        if hasattr(m, "model_dump"):
            result.append(m.model_dump())
        elif isinstance(m, dict) and "role" in m and "content" in m:
            result.append({"role": str(m["role"]), "content": str(m["content"])})
    return result

async def optimize_prompt_parallel(
    developer_message: str,
    messages: List["ChatMessage"],
) -> Dict[str, Any]:
    """
    Runs contradiction, format, and few-shot checkers in parallel,
    then rewrites the prompt/examples if needed.
    Returns a unified dict suitable for an API or endpoint.
    """

    with trace("optimize_prompt_workflow"):
        # 1. Run all checkers in parallel (contradiction, format, fewshot if there are examples)
        tasks = [
            Runner.run(dev_contradiction_checker, developer_message),
            Runner.run(format_checker, developer_message),
        ]
        if messages:
            fs_input = {
                "DEVELOPER_MESSAGE": developer_message,
                "USER_EXAMPLES": [m.content for m in messages if m.role == "user"],
                "ASSISTANT_EXAMPLES": [m.content for m in messages if m.role == "assistant"],
            }
            tasks.append(Runner.run(fewshot_consistency_checker, json.dumps(fs_input)))

        results = await asyncio.gather(*tasks)

        # Unpack results
        cd_issues: Issues = results[0].final_output
        fi_issues: Issues = results[1].final_output
        fs_issues: FewShotIssues = results[2].final_output if messages else FewShotIssues.no_issues()

        # 3. Rewrites as needed
        final_prompt = developer_message
        if cd_issues.has_issues or fi_issues.has_issues:
            pr_input = {
                "ORIGINAL_DEVELOPER_MESSAGE": developer_message,
                "CONTRADICTION_ISSUES": cd_issues.model_dump(),
                "FORMAT_ISSUES": fi_issues.model_dump(),
            }
            pr_res = await Runner.run(dev_rewriter, json.dumps(pr_input))
            final_prompt = pr_res.final_output.new_developer_message

        final_messages: list[ChatMessage] | list[dict[str, str]] = messages
        if fs_issues.has_issues:
            mr_input = {
                "NEW_DEVELOPER_MESSAGE": final_prompt,
                "ORIGINAL_MESSAGES": _normalize_messages(messages),
                "FEW_SHOT_ISSUES": fs_issues.model_dump(),
            }
            mr_res = await Runner.run(fewshot_rewriter, json.dumps(mr_input))
            final_messages = mr_res.final_output.messages

        return {
            "changes": True,
            "new_developer_message": final_prompt,
            "new_messages": _normalize_messages(final_messages),
            "contradiction_issues": "\n".join(cd_issues.issues),
            "few_shot_contradiction_issues": "\n".join(fs_issues.issues),
            "format_issues": "\n".join(fi_issues.issues),
        }

![工作流的追踪](../images/optimizepromptfig3.png)

### 理解优化工作流

`optimize_prompt_parallel` 函数实现了一个通过并行化最大化效率的工作流：

1. **并行检测问题**：第一阶段同时运行所有检查器智能体：
   - `dev_contradiction_checker` 查找提示中的逻辑矛盾
   - `format_checker` 查找不清晰的格式规范
   - `fewshot_consistency_checker`（如果存在示例）检查提示与示例之间的不匹配

在并行检查阶段之后，工作流会谨慎地处理依赖关系：

2. **重写提示（有条件）**：只有在检测到矛盾或格式问题时，`dev_rewriter` 智能体才会运行。该智能体依赖以下输出：
   - `dev_contradiction_checker`（`cd_issues` 变量）
   - `format_checker`（`fi_issues` 变量）

3. **重写示例（有条件）**：只有在检测到示例不一致时，`fewshot_rewriter` 智能体才会运行。该智能体依赖：
   - 重写后的提示（必须在提示重写之后执行）
   - 原始消息
   - 少样本问题（`fs_issues` 变量）


## 6. 示例

我们通过一些实际示例来看看优化系统的实际效果。

### 示例 1：修复矛盾

In [28]:
async def example_contradiction():
    # A prompt with contradictory instructions
    prompt = """Quick-Start Card — Product Parser

Goal  
Digest raw HTML of an e-commerce product detail page and emit **concise, minified JSON** describing the item.

**Required fields:**  
name | brand | sku | price.value | price.currency | images[] | sizes[] | materials[] | care_instructions | features[]

**Extraction priority:**  
1. schema.org/JSON-LD blocks  
2. <meta> & microdata tags  
3. Visible DOM fallback (class hints: "product-name", "price")

** Rules:**  
- If *any* required field is missing, short-circuit with: `{"error": "FIELD_MISSING:<field>"}`.
- Prices: Numeric with dot decimal; strip non-digits (e.g., "1.299,00 EUR" → 1299.00 + "EUR").
- Deduplicate images differing only by query string. Keep ≤10 best-res.
- Sizes: Ensure unit tag ("EU", "US") and ascending sort.
- Materials: Title-case and collapse synonyms (e.g., "polyester 100%" → "Polyester").

**Sample skeleton (minified):**
```json
{"name":"","brand":"","sku":"","price":{"value":0,"currency":"USD"},"images":[""],"sizes":[],"materials":[],"care_instructions":"","features":[]}
Note: It is acceptable to output null for any missing field instead of an error ###"""
    
    result = await optimize_prompt_parallel(prompt, [])
    
    # Display the results
    if result["contradiction_issues"]:
        print("Contradiction issues:")
        print(result["contradiction_issues"])
        print()
        
    print("Optimized prompt:")
    print(result["new_developer_message"])
    
# Run the example
await example_contradiction()

Contradiction issues:
There is a contradiction between the rule that says to short-circuit and output an error if *any* required field is missing ('{"error": "FIELD_MISSING:<field>"}') and the final note which states that it is acceptable to output null for any missing field instead of an error. Both behaviors cannot be followed simultaneously when a required field is missing.

Optimized prompt:
Quick-Start Card — Product Parser

Goal  
Digest raw HTML of an e-commerce product detail page and emit **concise, minified JSON** describing the item.

**Required fields:**  
name | brand | sku | price.value | price.currency | images[] | sizes[] | materials[] | care_instructions | features[]

**Extraction priority:**  
1. schema.org/JSON-LD blocks  
2. <meta> & microdata tags  
3. Visible DOM fallback (class hints: "product-name", "price")

**Rules:**  
- If *any* required field is missing, short-circuit and output: `{"error": "FIELD_MISSING:<field>"}`
- Prices: Numeric with dot decimal; strip

这展示了系统如何检测并解决可能导致输出不一致或让模型困惑的关键矛盾。 

### 示例 2：修复提示与少样本示例之间的不一致


In [29]:
async def example_fewshot_fix():
    prompt = "Respond **only** with JSON using keys `city` (string) and `population` (integer)."
    
    messages = [
        {"role": "user", "content": "Largest US city?"},
        {"role": "assistant", "content": "New York City"},
        {"role": "user", "content": "Largest UK city?"},
        {"role": "assistant", "content": "{\"city\":\"London\",\"population\":9541000}"}
    ]
    
    
    print("Few-shot examples before optimization:")
    print(f"User: {messages[0]['content']}")
    print(f"Assistant: {messages[1]['content']}")
    print(f"User: {messages[2]['content']}")
    print(f"Assistant: {messages[3]['content']}")
    print()
    
    # Call the optimization API
    result = await optimize_prompt_parallel(prompt, [ChatMessage(**m) for m in messages])
    
    # Display the results
    if result["few_shot_contradiction_issues"]:
        print("Inconsistency found:", result["few_shot_contradiction_issues"])
        print()
    
    # Show the optimized few-shot examples
    optimized_messages = result["new_messages"]
    print("Few-shot examples after optimization:")
    print(f"User: {optimized_messages[0]['content']}")
    print(f"Assistant: {optimized_messages[1]['content']}")
    print(f"User: {optimized_messages[2]['content']}")
    print(f"Assistant: {optimized_messages[3]['content']}")
    
# Run the example
await example_fewshot_fix()

Few-shot examples before optimization:
User: Largest US city?
Assistant: New York City
User: Largest UK city?
Assistant: {"city":"London","population":9541000}

Inconsistency found: The response 'New York City' does not use JSON format and is missing the required keys `city` and `population` as stated in the rule 'Respond **only** with JSON using keys `city` (string) and `population` (integer).'

Few-shot examples after optimization:
User: Largest US city?
Assistant: {"city":"New York City","population":8468000}
User: Largest UK city?
Assistant: {"city":"London","population":9541000}


这一点尤为重要，因为少样本示例对模型的回复方式有很大影响。如果示例不遵循既定规则，模型可能会学会忽略这些规则，转而模仿示例。通过确保提示指令与示例之间的一致性，优化系统可以生成更可靠的提示。

### 示例 3：在较长的提示中澄清格式

In [30]:
async def example_format_issue():
    # A prompt with unclear or inconsistent formatting instructions
    prompt = """Task → Translate dense patent claims into 200-word lay summaries with a glossary.

Operating Steps:
1. Split the claim at semicolons, "wherein", or numbered sub-clauses.
2. For each chunk:
   a) Identify its purpose.
   b) Replace technical nouns with everyday analogies.
   c) Keep quantitative limits intact (e.g., "≥150 C").
3. Flag uncommon science terms with asterisks, and later define them.
4. Re-assemble into a flowing paragraph; do **not** broaden or narrow the claim’s scope.
5. Omit boilerplate if its removal does not alter legal meaning.

Output should follow a Markdown template:
- A summary section.
- A glossary section with the marked terms and their definitions.

Corner Cases:
- If the claim is over 5 kB, respond with CLAIM_TOO_LARGE.
- If claim text is already plain English, skip glossary and state no complex terms detected.

Remember: You are *not* providing legal advice—this is for internal comprehension only."""

    # Call the optimization API to check for format issues
    result = await optimize_prompt_parallel(prompt, [])

    # Display the results
    if result.get("format_issues"):
        print("Format issues found:", result["format_issues"])
        print()

    print("Optimized prompt:")
    print(result["new_developer_message"])

# Run the example
await example_format_issue()

Format issues found: Output must follow a precise Markdown template, but the expected structure (headers, formatting) for the summary and glossary sections is not fully specified.
Ambiguity if output should be a Markdown string or a structured object containing Markdown—data type of output is implicit.
No explicit ordering instruction for the summary and glossary sections—potentially ambiguous.
Word count limit (200 words) is mentioned for the summary but not for the glossary section—scope unclear.
No specific format for CLAIM_TOO_LARGE error or for indicating 'no complex terms'—should these be Markdown or plaintext?

Optimized prompt:
Task → Translate dense patent claims into 200-word lay summaries with a glossary.

Operating Steps:
1. Split the claim at semicolons, "wherein", or numbered sub-clauses.
2. For each chunk:
   a) Identify its purpose.
   b) Replace technical nouns with everyday analogies.
   c) Keep quantitative limits intact (e.g., ">=150 C").
3. Flag uncommon science te

这个示例突出展示了格式检查器如何识别并解决含糊的格式规范。该提示要求输出 Markdown，而优化流程显著改进了这些格式规范。